# Tutorial: SQL for Data Science

> **Type:** Tutorial. Follow along step by step.  
> **Time:** ~2 hours  
> **Prerequisites:** Module 06 (pandas), basic SQL (SELECT, WHERE, JOIN)  
> **What you'll build:** An in-memory e-commerce database using DuckDB. You'll use it to answer 6 realistic business questions with GROUP BY, CTEs, and window functions.

---

## What We're Building

We'll use **DuckDB**, a fast, in-process analytical SQL engine that runs inside your Python session with no server setup. It reads straight from pandas DataFrames, which makes it ideal for data science work. We'll build a three-table e-commerce dataset (customers, orders, products), then write increasingly advanced SQL queries to answer realistic business questions.

## Step 0: Setup

In [ ]:
# !pip install duckdb pandas matplotlib seaborn

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

pd.set_option('display.max_columns', 15)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')
np.random.seed(42)

# Create an in-memory DuckDB connection
con = duckdb.connect()

print(f'DuckDB version: {duckdb.__version__}')
print('Ready.')

## Step 1: Build the E-Commerce Dataset

Three tables: customers, products, orders.

In [ ]:
# ── Customers ───────────────────────────────────────────────────────────────
n_customers = 200
regions = ['North', 'South', 'East', 'West']
tiers   = ['Bronze', 'Silver', 'Gold', 'Platinum']

customers = pd.DataFrame({
    'customer_id':    range(1, n_customers + 1),
    'name':          [f'Customer_{i}' for i in range(1, n_customers + 1)],
    'region':         np.random.choice(regions, n_customers),
    'tier':           np.random.choice(tiers, n_customers, p=[0.5, 0.3, 0.15, 0.05]),
    'signup_date':    pd.date_range('2020-01-01', periods=n_customers, freq='3D'),
})

# ── Products ────────────────────────────────────────────────────────────────
categories = ['Electronics', 'Clothing', 'Books', 'Home', 'Sports']
products = pd.DataFrame({
    'product_id':    range(1, 51),
    'name':         [f'Product_{i}' for i in range(1, 51)],
    'category':      np.random.choice(categories, 50),
    'price':         np.round(np.random.uniform(5, 500, 50), 2),
})

# ── Orders ──────────────────────────────────────────────────────────────────
n_orders = 3000
start_date = datetime(2022, 1, 1)
order_dates = [start_date + timedelta(days=int(d))
               for d in np.random.uniform(0, 730, n_orders)]

orders = pd.DataFrame({
    'order_id':     range(1, n_orders + 1),
    'customer_id':  np.random.choice(customers['customer_id'], n_orders),
    'product_id':   np.random.choice(products['product_id'], n_orders),
    'order_date':   order_dates,
    'quantity':     np.random.randint(1, 6, n_orders),
    'status':       np.random.choice(['completed','returned','pending'], n_orders,
                                     p=[0.85, 0.10, 0.05]),
})

# Add revenue (quantity × price)
price_map = dict(zip(products['product_id'], products['price']))
orders['revenue'] = orders.apply(
    lambda r: r['quantity'] * price_map[r['product_id']], axis=1
)

print(f'Customers: {len(customers):,}  |  Products: {len(products):,}  |  Orders: {len(orders):,}')
orders.head()

In [ ]:
# Register DataFrames as SQL tables in DuckDB
con.register('customers', customers)
con.register('products',  products)
con.register('orders',    orders)

# Helper: run SQL and return a DataFrame
def sql(query):
    return con.execute(query).df()

# Quick check
sql('SELECT COUNT(*) AS total_orders FROM orders')

---

## Query 1: Basic Aggregation — Revenue by Category

**Business question:** Which product category generates the most revenue?

In [ ]:
q1 = sql("""
    SELECT
        p.category,
        COUNT(o.order_id)          AS total_orders,
        ROUND(SUM(o.revenue), 2)   AS total_revenue,
        ROUND(AVG(o.revenue), 2)   AS avg_order_value
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
    WHERE o.status = 'completed'
    GROUP BY p.category
    ORDER BY total_revenue DESC
""")
q1

In [ ]:
# Visualise
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(q1['category'], q1['total_revenue'] / 1000,
              color='#4E79A7', edgecolor='white')
ax.set_title('Completed Revenue by Category', fontweight='bold')
ax.set_ylabel('Revenue ($k)')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'${bar.get_height():.0f}k', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('q1_revenue_by_category.png', dpi=120, bbox_inches='tight')
plt.show()

**Chart description.** A bar chart of completed revenue by product category, in thousands of dollars. Sports is highest at $630k, then clothing at $553k, home at $409k, electronics at $345k and books at $180k.

---

## Query 2: HAVING — Finding High-Value Customers

**Business question:** Which customers have spent more than $5,000 in total?

In [ ]:
q2 = sql("""
    SELECT
        c.customer_id,
        c.name,
        c.tier,
        c.region,
        COUNT(o.order_id)          AS orders,
        ROUND(SUM(o.revenue), 2)   AS lifetime_value
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    WHERE o.status = 'completed'
    GROUP BY c.customer_id, c.name, c.tier, c.region
    HAVING SUM(o.revenue) > 5000
    ORDER BY lifetime_value DESC
    LIMIT 10
""")
print(f'Customers with LTV > $5,000: shown top 10')
q2

---

## Query 3: CTE — Month-over-Month Revenue Growth

**Business question:** Is revenue growing month over month? By how much?

In [ ]:
q3 = sql("""
    -- Step 1: aggregate to monthly revenue
    WITH monthly AS (
        SELECT
            DATE_TRUNC('month', order_date)  AS month,
            ROUND(SUM(revenue), 2)           AS revenue
        FROM orders
        WHERE status = 'completed'
        GROUP BY 1
        ORDER BY 1
    ),
    -- Step 2: add previous month revenue using LAG
    with_lag AS (
        SELECT
            month,
            revenue,
            LAG(revenue) OVER (ORDER BY month) AS prev_revenue
        FROM monthly
    )
    -- Step 3: calculate growth
    SELECT
        month,
        revenue,
        prev_revenue,
        ROUND(revenue - prev_revenue, 2)                              AS abs_growth,
        ROUND((revenue - prev_revenue) / prev_revenue * 100, 1)      AS pct_growth
    FROM with_lag
    WHERE prev_revenue IS NOT NULL
""")
q3

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(q3['month'], q3['revenue'] / 1000, marker='o', color='#4E79A7', linewidth=2)
axes[0].set_title('Monthly Revenue', fontweight='bold')
axes[0].set_ylabel('Revenue ($k)')
axes[0].tick_params(axis='x', rotation=45)

colours = ['#59A14F' if v >= 0 else '#E15759' for v in q3['pct_growth']]
axes[1].bar(q3['month'], q3['pct_growth'], width=20, color=colours, edgecolor='white')  # width in days: the x-axis holds dates
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Month-over-Month Growth (%)', fontweight='bold')
axes[1].set_ylabel('Growth (%)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('q3_mom_growth.png', dpi=120, bbox_inches='tight')
plt.show()

**Chart description.** Left: monthly revenue from February 2022 to December 2023, in thousands of dollars. It moves between about $72k and $113k with no clear trend. The biggest swings are in late 2022: up to $104k in October, down to $73k in November and up to $113k in December. Right: month-over-month growth, green for growth and red for decline. Most months change by less than 10%. The extremes are +56% in December 2022 and −30% in November 2022.

---

## Query 4: Window Functions — Rank Customers Within Region

**Business question:** Who are the top 3 customers by lifetime value in each region?

In [ ]:
q4 = sql("""
    WITH customer_ltv AS (
        SELECT
            c.customer_id,
            c.name,
            c.region,
            c.tier,
            ROUND(SUM(o.revenue), 2) AS ltv
        FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id
        WHERE o.status = 'completed'
        GROUP BY c.customer_id, c.name, c.region, c.tier
    ),
    ranked AS (
        SELECT
            *,
            -- RANK within each region, ordered by LTV descending
            RANK() OVER (PARTITION BY region ORDER BY ltv DESC) AS rank_in_region
        FROM customer_ltv
    )
    SELECT * FROM ranked
    WHERE rank_in_region <= 3
    ORDER BY region, rank_in_region
""")
q4

> **What just happened?** `PARTITION BY region ORDER BY ltv DESC` tells the window function to restart ranking at 1 for each region, ordering by LTV highest-first. The outer query filters to the top 3 per region. Without `PARTITION BY`, `RANK()` would rank across all regions combined.

---

## Query 5: Running Total and Cumulative Percentage

**Business question:** What fraction of total revenue comes from our top N customers? (The Pareto question.)

In [ ]:
q5 = sql("""
    WITH customer_ltv AS (
        SELECT
            c.customer_id,
            ROUND(SUM(o.revenue), 2) AS ltv
        FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id
        WHERE o.status = 'completed'
        GROUP BY c.customer_id
    )
    SELECT
        customer_id,
        ltv,
        -- Running total (ROWS frame: tied ltv values don't share a total)
        SUM(ltv) OVER (ORDER BY ltv DESC, customer_id
                       ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total,
        -- Cumulative share of total revenue
        ROUND(
            SUM(ltv) OVER (ORDER BY ltv DESC, customer_id
                           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
            / SUM(ltv) OVER ()                         -- SUM OVER () = grand total
            * 100, 1
        )                                              AS cumulative_pct
    FROM customer_ltv
    LIMIT 20
""")
q5

In [ ]:
# Pareto chart: what % of revenue comes from top N customers?
all_customers = sql("""
    WITH customer_ltv AS (
        SELECT c.customer_id, SUM(o.revenue) AS ltv
        FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id
        WHERE o.status = 'completed'
        GROUP BY c.customer_id
    )
    SELECT
        ROW_NUMBER() OVER (ORDER BY ltv DESC, customer_id) AS customer_rank,
        ROUND(SUM(ltv) OVER (ORDER BY ltv DESC, customer_id
                             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
              / SUM(ltv) OVER () * 100, 2) AS cum_pct
    FROM customer_ltv
""")

# How many customers does it take to REACH each threshold?
# (first row where cum_pct >= threshold, not the last row still under it)
rank_80 = all_customers[all_customers['cum_pct'] >= 80].iloc[0]
rank_50 = all_customers[all_customers['cum_pct'] >= 50].iloc[0]
print(f"Customers needed to reach 80% of revenue: {int(rank_80['customer_rank'])} "
      f"({rank_80['customer_rank']/len(all_customers)*100:.1f}% of {len(all_customers)}), "
      f"cum_pct={rank_80['cum_pct']}")
print(f"Customers needed to reach 50% of revenue: {int(rank_50['customer_rank'])} "
      f"({rank_50['customer_rank']/len(all_customers)*100:.1f}% of {len(all_customers)}), "
      f"cum_pct={rank_50['cum_pct']}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(all_customers['customer_rank'], all_customers['cum_pct'],
        color='#4E79A7', linewidth=2)
ax.axhline(80, color='#E15759', linestyle='--', linewidth=1.5)
ax.axvline(rank_80['customer_rank'],
           color='#E15759', linestyle='--', linewidth=1.5)
ax.set_xlabel('Number of Customers (sorted by LTV)')
ax.set_ylabel('Cumulative % of Total Revenue')
ax.set_title('Pareto Curve: Customer Revenue Concentration', fontweight='bold')
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig('q5_pareto.png', dpi=120, bbox_inches='tight')
plt.show()

**Chart description.** A Pareto curve: the cumulative share of total revenue against the number of customers, sorted from highest lifetime value down. The curve is close to a straight line. Reaching 80% of revenue takes 138 of the 200 customers (69%), marked by dashed lines. Revenue isn't concentrated in a few customers.

---

## Query 6: End-to-End Business Question — Cohort Retention

**Business question:** of customers who first ordered in Q1 2022, what fraction placed a second order within 90 days?

This is a realistic, multi-step analytical question. It needs CTEs and window functions together.

In [ ]:
q6 = sql("""
    WITH first_orders AS (
        -- Find each customer's first order
        SELECT
            customer_id,
            MIN(order_date)  AS first_order_date
        FROM orders
        WHERE status = 'completed'
        GROUP BY customer_id
    ),
    q1_cohort AS (
        -- Cohort: customers whose first order was in Q1 2022
        SELECT customer_id, first_order_date
        FROM first_orders
        WHERE first_order_date >= '2022-01-01'
          AND first_order_date <  '2022-04-01'
    ),
    second_orders AS (
        -- Find their second order
        SELECT
            o.customer_id,
            o.order_date,
            ROW_NUMBER() OVER (PARTITION BY o.customer_id ORDER BY o.order_date) AS order_num
        FROM orders o
        JOIN q1_cohort c ON o.customer_id = c.customer_id
        WHERE o.status = 'completed'
          AND o.order_date > c.first_order_date
    )
    SELECT
        COUNT(DISTINCT q1_cohort.customer_id)           AS cohort_size,
        COUNT(DISTINCT CASE
            WHEN second_orders.order_num = 1
             AND second_orders.order_date
                  <= q1_cohort.first_order_date + INTERVAL '90 days'
            THEN second_orders.customer_id
        END)                                            AS retained_90d,
        ROUND(
            COUNT(DISTINCT CASE
                WHEN second_orders.order_num = 1
                 AND second_orders.order_date
                      <= q1_cohort.first_order_date + INTERVAL '90 days'
                THEN second_orders.customer_id
            END) * 100.0 / COUNT(DISTINCT q1_cohort.customer_id), 1
        )                                               AS retention_rate_pct
    FROM q1_cohort
    LEFT JOIN second_orders ON q1_cohort.customer_id = second_orders.customer_id
""")
q6

> **What just happened?** You built a cohort retention query from scratch using four CTEs. Analysts and data scientists write queries like this every week in production. The logic is complex, but the CTE structure gives each step a name and keeps it readable.

---

## Bringing SQL Results Back into pandas

In [ ]:
# Any SQL result is a pandas DataFrame: you can continue analysis in Python
result = sql("""
    SELECT p.category, c.region, ROUND(SUM(o.revenue), 2) AS revenue
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.status = 'completed'
    GROUP BY p.category, c.region
""")

# Pivot in pandas
pivot = result.pivot_table(values='revenue', index='category', columns='region', aggfunc='sum')

# Heatmap
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(pivot / 1000, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Revenue ($k)'})
ax.set_title('Revenue ($k) by Category × Region', fontweight='bold')
plt.tight_layout()
plt.savefig('sql_to_pandas_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

**Chart description.** A heatmap of revenue in thousands of dollars by category and region. Sports is strong everywhere, $146k to $166k. Clothing peaks in the West at $174k, the largest cell. Books is lowest everywhere, $39k to $50k. Electronics is highest in the East at $108k.

## What You Built

An in-memory analytical database with three tables, and six increasingly complex SQL queries: a basic GROUP BY, HAVING filters, CTE-based month-over-month growth, window-function ranking within groups, running totals for a Pareto curve, and a full cohort retention query. These patterns are used in production analytics every day.

## What's Next

- Read `explanation.md` for the theory behind window functions, and why CTEs are a readability tool, not a performance one.
- `howto.md` has ready-to-paste templates for the most common SQL patterns.
- `reference.md` is a window function syntax reference.
- **Extension challenge:** add a `discounts` table and another CTE to query 6 that tracks whether retained customers used a discount on their second order.

---

**Next module:** `09_exploratory_data_analysis`, where you learn to have a structured conversation with your data before modelling.